# 06 RAG Pipeline

**Final model stack:**

| Step | Model | Cost | Why |
|---|---|---|---|
| Embeddings (index) | `text-embedding-3-large` | ~$0.11 one-time | Best multilingual retrieval |
| Embeddings (query) | `text-embedding-3-large` | ~$0.000013/query | Same space as index |
| Query Rewriting | `gpt-4o-mini` | ~$0.000015/call | Cheap, fast, excellent instruction following |
| Answer Generation | `gpt-4o-mini` | ~$0.001/answer | Native Bulgarian, practical, direct |

**~$0.001 per conversation turn. Essentially free at personal use scale.**

**System prompt philosophy:**
- Direct, practical answers - not lectures
- Kilograms, not pounds
- Apply the advice immediately - no unnecessary citations
- Talk like a knowledgeable coach, not a textbook

**Pipeline:**
```
User question (BG)
        ↓
  Query Rewriting    ← gpt-4o-mini → optimized EN search query
        ↓
  Dual Retrieval     ← text-embedding-3-large + FAISS (local, free at query time)
        ↓
  Answer Generation  ← gpt-4o-mini → Bulgarian answer
```

In [14]:
import json, os
from pathlib import Path
from typing import Optional
import numpy as np
import faiss
import warnings; warnings.filterwarnings('ignore')

from openai import OpenAI
from dotenv import load_dotenv
from tenacity import retry, wait_exponential, stop_after_attempt
load_dotenv(override=True)

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
VS_DIR = BACKEND_DIR / 'data' / 'vectorstore'

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

EMBED_MODEL = os.getenv('EMBEDDING_MODEL')
EMBED_DIM = 3072
LLM_MODEL = os.getenv('PRIMARY_MODEL')
RETRIEVAL_K = 20
FINAL_K = 5
TEMPERATURE = 0.4

print('Setup complete')
print(f'Embedding: {EMBED_MODEL}')
print(f'LLM: {LLM_MODEL}')

Setup complete
Embedding: text-embedding-3-large
LLM: gpt-4o-mini


## 1. Load FAISS Index + Metadata

In [15]:
index = faiss.read_index(str(VS_DIR / 'henselmans_openai.index'))
metadata = json.loads((VS_DIR / 'henselmans_openai_metadata.json').read_text(encoding='utf-8'))
assert index.ntotal == len(metadata)
print(f'FAISS index: {index.ntotal:,} vectors  ({EMBED_DIM}d)')
print(f'Metadata: {len(metadata):,} records')

FAISS index: 2,098 vectors  (3072d)
Metadata: 2,098 records


## 2. Query Rewriting

In [16]:
@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def rewrite_query(question: str, history: list[dict] = []) -> str:
    """Rewrite any-language question to optimized EN search query for the Henselmans KB."""
    history_ctx = ''
    if history:
        for msg in history[-4:]:
            role = 'User' if msg['role'] == 'user' else 'Assistant'
            history_ctx += f'{role}: {msg["content"][:150]}\n'

    prompt = f"""{f'Recent conversation:{chr(10)}{history_ctx}{chr(10)}' if history_ctx else ''}User question: {question}

Rewrite as a SHORT English search query (5-10 words) for a fitness science knowledge base (Henselmans PTC course).
Reply ONLY with the search query, nothing else."""

    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=30,
    )
    return response.choices[0].message.content.strip().strip('"')


# Test
q_test = 'Колко протеин трябва да приемам за мускулна маса?'
rewritten = rewrite_query(q_test)
print(f'Original: {q_test}')
print(f'Rewritten: {rewritten}')

Original: Колко протеин трябва да приемам за мускулна маса?
Rewritten: How much protein for muscle mass?


## 3. Dual Retrieval

In [17]:
def embed_query(text: str) -> np.ndarray:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=[text])
    emb = np.array(r.data[0].embedding, dtype='float32').reshape(1, -1)
    faiss.normalize_L2(emb)
    return emb

def retrieve_faiss(query: str, k: int = RETRIEVAL_K) -> list:
    q_emb = embed_query(query)
    scores, ids = index.search(q_emb, k)
    return [
        {**metadata[idx], 'score': float(s)}
        for s, idx in zip(scores[0], ids[0]) if idx != -1
    ]

def mmr_diversity(candidates: list, top_n: int = FINAL_K, lambda_: float = 0.6) -> list:
    """
    Maximal Marginal Relevance - balance relevance vs diversity.
    Prevents returning 5 chunks from the same file.
    lambda_: 0=max diversity, 1=max relevance. 0.6 = good balance.
    """
    if len(candidates) <= top_n:
        return candidates

    selected = [candidates[0]]
    remaining = candidates[1:]

    while len(selected) < top_n and remaining:
        best_score = -999
        best_idx = 0

        for j, cand in enumerate(remaining):
            # Relevance score (already computed by FAISS)
            relevance = cand['score']

            # Diversity: penalize if similar source already selected
            same_source_count = sum(1 for s in selected if s['source'] == cand['source'])
            diversity_penalty = same_source_count * 0.15   # -0.15 per duplicate source

            mmr_score = lambda_ * relevance - (1 - lambda_) * diversity_penalty

            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = j

        selected.append(remaining.pop(best_idx))

    return selected

def dual_retrieve(original: str, rewritten: str, k: int = RETRIEVAL_K) -> list:
    """
    Retrieve with both queries, merge, deduplicate, apply MMR for diversity.
    """
    seen = {}
    for r in retrieve_faiss(original, k) + retrieve_faiss(rewritten, k):
        cid = r['chunk_id']
        if cid not in seen or r['score'] > seen[cid]['score']:
            seen[cid] = r
    merged = sorted(seen.values(), key=lambda x: x['score'], reverse=True)
    return merged

# Test
print('Retrieval functions defined with MMR diversity')
_q = 'protein intake while cutting'
_r = dual_retrieve(_q, _q)
print(f'Candidates: {len(_r)}')
sources = [r["source"] for r in _r[:8]]
print(f'Top-8 sources: {sources}')

Retrieval functions defined with MMR diversity
Candidates: 20
Top-8 sources: ['Protein PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Ad libitum dieting case study.pdf', 'Protein PTC 2022.pdf', 'Ad libitum dieting PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Protein PTC 2022.pdf']


## 4. Answer Generation

In [18]:
SYSTEM_PROMPT = """Ти си персонален фитнес треньор и нутриционист. Работиш изцяло по методологията на Menno Henselmans.

ПРАВИЛА - следвай ги стриктно:
1. Отговаряй ВИНАГИ на БЪЛГАРСКИ
2. Ползвай КИЛОГРАМИ и метри, никога pounds или inches
3. Бъди ДИРЕКТЕН и ПРАКТИЧЕН - давай конкретни числа и препоръки
4. НЕ изнасяй лекции и НЕ цитирай проучвания, ако не са поискани
5. Говори като треньор, не като учебник - просто, ясно, приложимо
6. Ако нещо не е покрито в контекста, кажи го честно
7. Адаптирай отговора спрямо нивото и целта на потребителя
8. Посочвай винаги имената на упражненията единствено и САМО на английски език
9. Задължително базирай всичките си отговори САМО на ресурсите от курса

Контекст от курса на Henselmans (използвай САМО тази информация за факти):
{context}"""


@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def generate_answer(question: str, chunks: list[dict], history: list[dict] = []) -> str:
    """Generate Bulgarian answer with gpt-4o-mini using retrieved context."""
    context = '\n\n'.join(
        f'[{c["source"]}]\n{c["text"]}' for c in chunks
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT.format(context=context)}]

    for msg in history:
        messages.append({'role': msg['role'], 'content': msg['content']})

    messages.append({'role': 'user', 'content': question})

    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=1200,
    )
    return response.choices[0].message.content


# Test
answer = generate_answer(q_test, candidates[:FINAL_K])
print(f'Q: {q_test}')
print()
print(answer)

Q: Колко протеин трябва да приемам за мускулна маса?

За оптимален растеж на мускулна маса, препоръчителният прием на протеин е 1.6 г на килограм телесно тегло на ден. Ако целта ти е да увеличиш мускулната маса, можеш да се стремиш към 1.8 г на килограм телесно тегло. 

Например, ако тежиш 70 кг, целевият ти прием на протеин би бил между 112 г и 126 г на ден. Раздели този прием на 3-4 хранения, за да осигуриш постоянен поток от аминокиселини.


## 5. Full Pipeline

In [19]:
RETRIEVAL_K = 20
FINAL_K = 7    # increased from 5 for better recall
TEMPERATURE = 0.4

def ask(
    question: str,
    history: list = [],
    filter_meta: dict = None,
) -> tuple:
    """
    Full RAG pipeline: question -> Bulgarian answer.
    Uses MMR diversity to avoid returning chunks from single source.
    """
    rewritten = rewrite_query(question, history)
    candidates = dual_retrieve(question, rewritten, k=RETRIEVAL_K)

    if filter_meta:
        filtered = [c for c in candidates if all(c.get(k)==v for k,v in filter_meta.items())]
        candidates = filtered or candidates

    # Apply MMR for diverse top chunks
    top_chunks = mmr_diversity(candidates[:30], top_n=FINAL_K, lambda_=0.6)

    answer = generate_answer(question, top_chunks, history)
    return answer, top_chunks

print('ask() defined - FINAL_K=7, MMR diversity enabled')

ask() defined - FINAL_K=7, MMR diversity enabled


## 6. Tests

In [20]:
print('=' * 60)
print('TEST 1: Protein intake on a cut')
print('=' * 60)
answer, chunks = ask('Колко протеин трябва да приемам докато съм на дефицит?')
print(f'Sources: {[c["source"][:40] for c in chunks]}')
print()
print(answer)

TEST 1: Protein intake on a cut
Sources: ['Protein PTC 2022.pdf', 'Ad libitum dieting PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Energy PTC 2022.pdf', 'Case study female Powerlifter prep.pdf', 'Protein PTC 2022.pdf', 'Energy PTC 2022.pdf']

Докато си на дефицит, трябва да приемаш между 1.2 и 1.6 г протеин на килограм телесно тегло на ден. Например, ако тежиш 70 кг, това означава, че трябва да приемаш между 84 и 112 г протеин дневно. 

Ако си по-напреднал трениращ, можеш да увеличиш приема до 1.8 г на килограм. Важно е да следиш и как се чувстваш, както и да адаптираш приема на протеин спрямо индивидуалните си нужди и цели.


In [21]:
print()
print('=' * 60)
print('TEST 2: Training frequency')
print('=' * 60)
answer, chunks = ask('Колко пъти в седмицата трябва да тренирам всяка мускулна група?')
print()
print(answer)


TEST 2: Training frequency

Всяка мускулна група трябва да се тренира поне два пъти в седмицата. Ако обемът на тренировките ти надвишава 16 серии на мускулна група, е препоръчително да я тренираш поне три пъти в седмицата. За напреднали атлети, които увеличават обема на тренировките си, може да бъде полезно да тренират всяка мускулна група ежедневно или дори два пъти на ден.


In [22]:
print()
print('=' * 60)
print('TEST 3: Multi-turn conversation')
print('=' * 60)
history = []

q1 = 'Какво е прогресивно претоварване?'
a1, _ = ask(q1)
history += [{'role': 'user', 'content': q1}, {'role': 'assistant', 'content': a1}]
print(f'Q1: {q1}')
print(f'A1: {a1[:300]}')
print()

q2 = 'Как да го приложа практически в тренировките си?'
a2, _ = ask(q2, history=history)
history += [{'role': 'user', 'content': q2}, {'role': 'assistant', 'content': a2}]
print(f'Q2: {q2}')
print(f'A2: {a2[:300]}')
print()

q3 = 'С колко килограма да увеличавам теглото всяка седмица във фитнеса?'
a3, _ = ask(q3, history=history)
print(f'Q3: {q3}')
print(f'A3: {a3[:300]}')


TEST 3: Multi-turn conversation
Q1: Какво е прогресивно претоварване?
A1: Прогресивното претоварване е принцип в тренировките, който означава постепенно увеличаване на стреса върху мускулите, за да се стимулира тяхното развитие и адаптация. Това може да се постигне чрез:

1. Увеличаване на тежестта, която вдигате.
2. Увеличаване на броя на повторенията, които изпълнявате 

Q2: Как да го приложа практически в тренировките си?
A2: За да приложиш прогресивното претоварване практически в тренировките си, следвай тези стъпки:

1. **Определи целите си**: Реши дали искаш да увеличиш силата, мускулната маса или издръжливостта.

2. **Започни с подходяща тежест**: Избери тежест, с която можеш да изпълниш целевия брой повторения (напр

Q3: С колко килограма да увеличавам теглото всяка седмица във фитнеса?
A3: При прогресивното претоварване, увеличаването на тежестта зависи от твоята текуща сила и опит. За начинаещи, ето как можеш да подходиш:

1. **Начинаещи**: Увеличавай тежестта с 1-2.5 кг на 

In [23]:
print()
print('=' * 60)
print('TEST 4: Program design question')
print('=' * 60)
answer, chunks = ask(
    'Трябва ми примерна програма за напреднал състезател 3 пъти седмично',
    filter_meta={'is_case_study': True}
)
print(f'Case study sources:')
for c in chunks:
    print(f'  {c["source"][:55]}')
print()
print(answer)


TEST 4: Program design question
Case study sources:
  Case study average Joe program design.pdf
  Case study Chad BJJ.pdf
  Case study untrained.pdf
  Case study novice vegan.pdf
  Case study block periodization and mixing different goa
  Case study Chad BJJ.pdf
  Case study novice vegan.pdf

Разбира се! Ето примерна програма за напреднал състезател, която се тренира 3 пъти седмично. Програмата е разделена на А и Б дни, които се редуват.

**Ден A:**

1. **Squat**: 4 серии x 6-8 повторения
2. **Bench Press**: 4 серии x 6-8 повторения
3. **Bent Over Row**: 4 серии x 6-8 повторения
4. **Overhead Press**: 3 серии x 8-10 повторения
5. **Leg Press**: 3 серии x 10-12 повторения
6. **Tricep Dips**: 3 серии x 8-10 повторения
7. **Plank**: 3 серии x 30-60 секунди

**Ден Б:**

1. **Deadlift**: 4 серии x 6-8 повторения
2. **Pull-Up**: 4 серии x 6-8 повторения
3. **Incline Dumbbell Press**: 4 серии x 8-10 повторения
4. **Leg Curl**: 3 серии x 10-12 повторения
5. **Lateral Raise**: 3 серии x 10-12 

## 7. Summary

In [24]:
print('=' * 60)
print('  NOTEBOOK 06 - COMPLETE (OpenAI Stack)')
print('=' * 60)
print()
print('  Stack:')
print(f'  Embeddings: text-embedding-3-large (3072d)')
print(f'  LLM: gpt-4o-mini')
print(f'  Retrieval: FAISS IndexFlatIP (exact cosine)')
print()
print('  System prompt: direct, practical, Bulgarian,')
print('  kilograms, coach-style - no unnecessary citations')

  NOTEBOOK 06 - COMPLETE (OpenAI Stack)

  Stack:
  Embeddings: text-embedding-3-large (3072d)
  LLM: gpt-4o-mini
  Retrieval: FAISS IndexFlatIP (exact cosine)

  System prompt: direct, practical, Bulgarian,
  kilograms, coach-style - no unnecessary citations
